In [8]:
#a,Thuật toán AKT vào bài toán 8 puzzle(n=3)
import copy
from heapq import heappush, heappop

# Giả sử puzzle(n=3) n = 3
n = 3 # Explicitly define n
# 4 vị trí dịch chuyền tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi
class priorityQueue:
  def __init__(self): # Corrected __init__
    self.heap = []
  # Inserting a new key 'key'
  def push(self, key):
      heappush (self.heap, key)
  #funct to remove the element that is min from the Priority Queue
  def pop(self):
      return heappop(self.heap)
  # funct to check if the Queue is empty or not
  def empty(self):
      if not self.heap:
          return True
      else:
          return False

# structure of the node
class nodes:
  def __init__(self, parent, mats, empty_tile_posi, costs, levels): # Corrected __init__
      self.parent = parent
      # Useful for Storing the matrix
      self.mats = mats
      self.empty_tile_posi = empty_tile_posi
      self.costs = costs
      self.levels = levels

  def __lt__ (self, nxt): # Corrected __lt__
      return self.costs < nxt.costs

def calculateCosts (mats, final) -> int:
  count = 0
  for i in range(n):
      for j in range(n):
          # Exclude empty tile (0) from cost calculation
          if ((mats[i][j] != 0) and (mats[i][j] != final[i][j])): count += 1
  return count

def newNodes (mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
   # Copying data from the parent matrixes to the present matrixes
   new_mats = copy.deepcopy (mats)
   # Moving the tile by 1 position
   x1 = empty_tile_posi[0]
   y1 = empty_tile_posi[1]
   x2 = new_empty_tile_posi[0]
   y2 = new_empty_tile_posi[1]
   new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats [x1][y1]
   # Setting the no. of misplaced tiles
   costs = calculateCosts(new_mats, final)
   new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
   return new_nodes

# func to print the N by N matrix
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()

# func to know if (x, y) is a valid or invalid
# matrix coordinates
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Printing the path from the root node to the final node
def printPath(root):
    if root == None:
        return
    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()
    # Creating the root node
    costs = calculateCosts (initial, final)
    root = nodes (None, initial, empty_tile_posi, costs, 0)
    # Adding root to the list of live nodes
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()
        if minimum.costs == 0:
            printPath(minimum)
            return

        # Generating all feasible children
        for i in range(4): # 4 possible moves (bottom, left, top, right)
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]
            if isSafe(new_x, new_y):
                new_empty_tile_posi = [new_x, new_y]
                child = newNodes(minimum.mats, minimum.empty_tile_posi, new_empty_tile_posi,
                                 minimum.levels + 1, minimum, final)
                pq.push(child)
                new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]
            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                # Creating a child node
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1,
                                 minimum, final,)
                # Adding the child to the list of live nodes
                pq.push(child)

# Initial configuration for 8-puzzle
initial = [ [ 1, 2, 3 ],
            [ 5, 6, 0 ],
            [ 7, 8, 4 ] ]
# Final configuration for 8-puzzle
final = [ [ 1, 2, 3 ],
          [ 5, 8, 6 ],
          [ 0, 7, 4 ] ]

# Position of the empty tile (0) in the initial state
empty_tile_posi = [1, 2]
solve(initial, empty_tile_posi, final)
#b,Thuật toán tìm kiếm A* trên đồ thị
from collections import deque

class Graph:
  def __init__ (self, adjac_lis): # Corrected __init__
    self.adjac_lis = adjac_lis

  def get_neighbors(self, v):
      return self.adjac_lis[v]

  # This is heuristic function which is having equal values for all nodes
  def h(self, n):
    H = {
        'A': 1,
        'B': 1,
        'C': 1,
        'D': 1
    }
    return H[n]

  def a_star_algorithm(self, start, stop):
     open_1st = set([start])
     closed_1st = set([])

     poo = {} # Actual cost from start node to current node n
     poo[start] = 0

     # par contains an adjacency mapping of all nodes for path reconstruction
     par = {}
     par[start] = start

     while len(open_1st) > 0:
      n = None

      # It will find a node with the lowest value of f() = g(n) + h(n)
      for v in open_1st:
        if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
          n = v;

      if n == None:
        print('Path does not exist!')
        return None

      # If the current node is the stop node, reconstruct and return the path
      if n == stop:
        reconst_path = []

        while par[n] != n:
          reconst_path.append(n)
          n = par[n]

        reconst_path.append(start)
        reconst_path.reverse()

        print('Path found: {}'.format(reconst_path))
        return reconst_path

      # For all the neighbors of the current node
      for (m, weight) in self.get_neighbors(n):
        # If the current node is not present in both open_1st and closed_1st
        # add it to open_1st and note n as its par
        if m not in open_1st and m not in closed_1st:
          open_1st.add(m)
          par[m] = n
          poo[m] = poo[n] + weight
        else:
            # If it is in open_1st or closed_1st, check if new path is better
            if poo[m] > poo[n] + weight:
                poo[m] = poo[n] + weight
                par[m] = n

                if m in closed_1st:
                  closed_1st.remove(m)
                  open_1st.add(m)

      open_1st.remove(n)
      closed_1st.add(n)

     print('Path does not exist!')
     return None
adjac_lis = {
      'A':[('B', 1), ('C', 3), ('D', 7)],
      'B':[('C',1), ('D', 5)],
      'C': [('D',12)]
     }

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')



1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  

Path found: ['A', 'B', 'C']


['A', 'B', 'C']

In [17]:
import copy
from heapq import heappush, heappop

n = 4

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class Node:
    def __init__(self, parent, mats, empty_pos, g, h):
        self.parent = parent
        self.mats = mats
        self.empty_pos = empty_pos
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f


def calculateCost(mats, final):
    count = 0
    for i in range(n):
        for j in range(n):
            if mats[i][j] != 0 and mats[i][j] != final[i][j]:
                count += 1
    return count


def newNode(mats, empty_pos, new_pos, g, parent, final):
    new_mats = copy.deepcopy(mats)

    x1, y1 = empty_pos
    x2, y2 = new_pos

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    h = calculateCost(new_mats, final)

    return Node(parent, new_mats, new_pos, g, h)


def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n


def printMatrix(mats):
    for i in range(n):
        for j in range(n):
            print(mats[i][j], end=" ")
        print()
    print()


def printPath(node):
    if node is None:
        return
    printPath(node.parent)
    printMatrix(node.mats)


def matToTuple(mat):
    return tuple(tuple(row) for row in mat)


def solve(initial, empty_pos, final):
    pq = []
    visited = set()

    h = calculateCost(initial, final)
    root = Node(None, initial, empty_pos, 0, h)

    heappush(pq, root)

    while pq:
        current = heappop(pq)

        if current.h == 0:
            print("Solution:\n")
            printPath(current)
            return

        visited.add(matToTuple(current.mats))

        for i in range(4):
            new_x = current.empty_pos[0] + rows[i]
            new_y = current.empty_pos[1] + cols[i]

            if isSafe(new_x, new_y):
                child = newNode(current.mats,
                                current.empty_pos,
                                [new_x, new_y],
                                current.g + 1,
                                current,
                                final)

                if matToTuple(child.mats) not in visited:
                    heappush(pq, child)


# TEST 15 puzzle
initial = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9,10, 7,11],
    [13,14,15,12]
]

final = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9,10,11,12],
    [13,14,15, 0]
]

solve(initial, [1,2], final)

Solution:

1 2 3 4 
5 6 0 8 
9 10 7 11 
13 14 15 12 

1 2 3 4 
5 6 7 8 
9 10 0 11 
13 14 15 12 

1 2 3 4 
5 6 7 8 
9 10 11 0 
13 14 15 12 

1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 0 



In [19]:
#Cau2
class Graph:
    def __init__(self, adjac_lis):
        self.adjac_lis = adjac_lis

    def get_neighbors(self, v):
        return self.adjac_lis[v]

    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):
        open_set = set([start])
        closed_set = set()

        g = {}
        g[start] = 0

        parent = {}
        parent[start] = start

        while open_set:
            n = None

            for v in open_set:
                if n is None or g[v] + self.h(v) < g[n] + self.h(n):
                    n = v

            if n is None:
                print("Không tìm thấy đường")
                return None

            if n == stop:
                path = []
                while parent[n] != n:
                    path.append(n)
                    n = parent[n]
                path.append(start)
                path.reverse()

                print("Đường đi:", path)
                return path

            for (m, weight) in self.get_neighbors(n):
                if m not in open_set and m not in closed_set:
                    open_set.add(m)
                    parent[m] = n
                    g[m] = g[n] + weight
                else:
                    if g[m] > g[n] + weight:
                        g[m] = g[n] + weight
                        parent[m] = n

                        if m in closed_set:
                            closed_set.remove(m)
                            open_set.add(m)

            open_set.remove(n)
            closed_set.add(n)

        print("Không có đường đi")
        return None


# TEST
adjac_lis = {
    'A':[('B', 1), ('C', 3), ('D', 7)],
    'B':[('C',1), ('D', 5)],
    'C':[('D',12)],
    'D':[]
}

graph = Graph(adjac_lis)
graph.a_star_algorithm('A', 'D')

Đường đi: ['A', 'B', 'D']


['A', 'B', 'D']